# OCR Detection (Multi-PDF) — Azure OpenAI

## Activate the packages

In [1]:
import base64
import os
from pathlib import Path

import pymupdf
from openai import AzureOpenAI
from dotenv import load_dotenv

## Load the credential from the .env file

Uses Azure OpenAI keys from `.env`:
- `AZURE_OPENAI_API_KEY`
- `AZURE_OPENAI_ENDPOINT`
- `AZURE_OPENAI_DEPLOYMENT_NAME`
- `AZURE_OPENAI_API_VERSION`

In [ ]:
# Always load project-root .env (works even if notebook cwd differs)
ROOT = Path.cwd()
if not (ROOT / ".env").exists():
    ROOT = Path.cwd().resolve()
load_dotenv(ROOT / ".env", override=True)
print("Loaded .env from:", (ROOT / ".env").resolve() if (ROOT / ".env").exists() else "NOT FOUND")

Loaded .env from: C:\Users\VINCENT_WF_MAN\Documents\Training\VideCoding_FEVAWORK\Class four\OCR_Engine_Portable\.env


## Config

- Put all PDF files you want to process into one folder
- Or list specific PDF paths in `PDF_FILES`
- Each PDF will produce its own Markdown file: `{pdf_name}.md`

In [3]:
# Option A: process all PDFs in a folder
PDF_FOLDER = "."

In [23]:
# Option B: process only specific files (leave empty list to use folder mode)
PDF_FILES = []

In [4]:
DPI = int(os.getenv("OCR_DPI", "200"))
MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME") or os.getenv("OCR_MODEL", "gpt-4o")
PROMPT = "Extract all text from this image. Output clean Markdown only."

print(f"DPI={DPI}")
print(f"MODEL (deployment)={MODEL}")

DPI=200
MODEL (deployment)=gpt-4o


## Azure OpenAI API Connection

In [5]:
api_key = os.getenv("AZURE_OPENAI_API_KEY")
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")

if not api_key or not endpoint:
    raise RuntimeError(
        "Missing AZURE_OPENAI_API_KEY or AZURE_OPENAI_ENDPOINT in .env"
    )

# Azure AI Foundry project URLs (.../api/projects/...) break AzureOpenAI chat;
# use the resource root instead (e.g. https://<resource>.services.ai.azure.com)
if "/api/projects/" in endpoint:
    endpoint = endpoint.split("/api/projects/")[0]
    print("Normalized endpoint (stripped /api/projects/...):", endpoint)

client = AzureOpenAI(
    api_key=api_key,
    azure_endpoint=endpoint,
    api_version=api_version,
)

print(f"Endpoint: {endpoint}")
print(f"API version: {api_version}")
print(f"Deployment: {MODEL}")

Endpoint: https://vincentman1027-4220-resource.services.ai.azure.com
API version: 2024-10-XX
Deployment: gpt-4o


## Helper functions

In [8]:
def pdf_to_png_bytes(pdf_path: str | Path, dpi: int = 200) -> list[bytes]:
    """Render each PDF page to PNG bytes."""
    zoom = dpi / 72.0
    matrix = pymupdf.Matrix(zoom, zoom)
    pages = []
    with pymupdf.open(pdf_path) as doc:
        for page in doc:
            pix = page.get_pixmap(matrix=matrix, alpha=False)
            pages.append(pix.tobytes("png"))
    return pages

In [9]:
def ocr_png_bytes(png_bytes: bytes, model: str, prompt: str) -> str:
    """Send one page image to Azure OpenAI vision and return extracted text."""
    b64 = base64.b64encode(png_bytes).decode()
    data_url = f"data:image/png;base64,{b64}"

    resp = client.chat.completions.create(
        model=model,  # Azure deployment name, e.g. gpt-4o
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": data_url}},
            ],
        }],
    )
    return (resp.choices[0].message.content or "").strip()

In [10]:
def ocr_one_pdf(pdf_path: Path, model: str, prompt: str, dpi: int) -> Path:
    """
    OCR one PDF and save result as {pdf_stem}.md in the same folder.
    Returns the path of the written Markdown file.
    """
    print(f"\n=== Processing: {pdf_path.name} ===")
    pages = pdf_to_png_bytes(pdf_path, dpi)
    results = []

    for i, png in enumerate(pages, 1):
        print(f"  OCR page {i}/{len(pages)} ...")
        text = ocr_png_bytes(png, model=model, prompt=prompt)
        if len(pages) == 1:
            results.append(text)
        else:
            results.append(f"## Page {i}\n\n{text}")

    # Save as separate MD file next to the PDF
    output_path = pdf_path.with_suffix(".md")   # e.g. invoice.pdf -> invoice.md
    output_path.write_text("\n\n".join(results), encoding="utf-8")
    print(f"  Saved: {output_path}")
    return output_path

## Collect PDF list & run OCR

In [14]:
# Build the list of PDFs to process
if PDF_FILES:
    pdf_list = [Path(p) for p in PDF_FILES]
else:
    folder = Path(PDF_FOLDER)
    pdf_list = sorted(folder.glob("*.pdf"))

print(f"Found {len(pdf_list)} PDF(s) to process:")
for p in pdf_list:
    print(f"  - {p}")

Found 3 PDF(s) to process:
  - test_invoice.pdf
  - test_invoice2.pdf
  - test_invoice_3.pdf


In [15]:
saved_files = []

for pdf_path in pdf_list:
    if not pdf_path.is_file():
        print(f"SKIP (not found): {pdf_path}")
        continue
    try:
        out = ocr_one_pdf(pdf_path, model=MODEL, prompt=PROMPT, dpi=DPI)
        saved_files.append(out)
    except Exception as e:
        print(f"ERROR on {pdf_path.name}: {e}")

print("\n===== Done =====")
print(f"Successfully processed {len(saved_files)} file(s):")
for f in saved_files:
    print(f"  -> {f}")


=== Processing: test_invoice.pdf ===
  OCR page 1/1 ...
  Saved: test_invoice.md

=== Processing: test_invoice2.pdf ===
  OCR page 1/1 ...
  Saved: test_invoice2.md

=== Processing: test_invoice_3.pdf ===
  OCR page 1/1 ...
  Saved: test_invoice_3.md

===== Done =====
Successfully processed 3 file(s):
  -> test_invoice.md
  -> test_invoice2.md
  -> test_invoice_3.md
